# 06 - Training Loop

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 训练步骤地图

| 步骤 | 代码 | 你要盯住什么 |
|---|---|---|
| forward | `loss()` | 当前预测和 loss |
| 清空旧梯度 | `model.zero_grad()` | 所有 `p.grad` 回到 0 |
| backward | `total_loss.backward()` | 每个参数得到新 `grad` |
| update | `p.data += -lr * p.grad` | 参数朝降低 loss 的方向动一点 |
| repeat | `for step in range(...)` | loss 整体下降 |

这张表就是训练循环的骨架。

## 1. 一个很小的数据集

输入是两个数字，目标输出是 `1` 或 `-1`。我们让 MLP 学会把输入映射到目标。

In [ ]:
xs = [
    [2.0, 3.0],
    [3.0, -1.0],
    [0.5, 1.0],
    [1.0, 1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

for x, y in zip(xs, ys):
    print(x, '->', y)

## 2. 建一个小 MLP

`MLP(2, [4, 1])` 的意思：

```text
输入维度 2
隐藏层 4 个神经元
输出层 1 个神经元
```

In [ ]:
random.seed(1337)
model = MLP(2, [4, 1])
print(model)
print('number of parameters:', len(model.parameters()))

## 3. 定义 loss：先把“错多少”变成一个数

训练前，模型只是随机猜。我们需要一个数字告诉它：**现在错得有多严重**。这个数字就是 `loss`。

这一节用最简单的平方误差。先不要看公式，先看一条样本：

```text
标准答案 ygt = 1.0
模型预测 yout = 0.4
误差 diff = yout - ygt = 0.4 - 1.0 = -0.6
平方误差 squared = diff ** 2 = (-0.6) ** 2 = 0.36
```

为什么要平方？

```text
预测偏大：diff 是正数
预测偏小：diff 是负数
平方以后都变成正数
错得越远，平方误差越大
```

多条样本时，就对每条都算一次平方误差，然后取平均：

$$
loss = \frac{1}{N}\sum_i (\hat{y}_i - y_i)^2
$$

把符号翻译成人话：

| 符号/变量 | 含义 |
|---|---|
| `ygt` | ground truth，标准答案 |
| `yout` | 模型输出的预测值 |
| `yout - ygt` | 预测和答案差多少 |
| `(yout - ygt) ** 2` | 这一条样本的平方误差 |
| `sum(losses) / len(losses)` | 所有样本的平均错误，也就是总 loss |




In [ ]:
# 先用普通 Python 数字算一条样本，不涉及 Value 和 backward。
toy_ygt = 1.0      # 标准答案
toy_yout = 0.4     # 模型预测

toy_diff = toy_yout - toy_ygt
toy_squared = toy_diff ** 2

print('ygt      =', toy_ygt)
print('yout     =', toy_yout)
print('diff     = yout - ygt =', toy_diff)
print('squared  = diff ** 2  =', toy_squared)



### 把一条样本扩展到 4 条样本

当前数据集里有 4 条训练样本：

```python
xs = [
    [2.0, 3.0],
    [3.0, -1.0],
    [0.5, 1.0],
    [1.0, 1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]
```

`loss()` 做的事情就是：

```text
1. 对每条 x 调一次 predict(x)，得到 ypred
2. 把 ypred 和 ys 一一配对
3. 每一对都算 (预测值 - 标准答案) ** 2
4. 把 4 个平方误差求平均
```

注意：这里还没有更新参数，只是在问当前模型“现在错多少”。



In [ ]:
def predict(xrow):
    # xrow 是普通 Python 数字列表，比如 [2.0, 3.0]
    # model 需要吃 Value，所以先把每个数字包成 Value。
    values = [Value(x) for x in xrow]
    return model(values)


def loss():
    # 1. 对每条输入都做一次预测。
    ypred = [predict(x) for x in xs]

    # 2. 把标准答案 ys 和预测值 ypred 一一配对。
    # 3. 每条样本都算一个平方误差。
    losses = [(yout - ygt) ** 2 for ygt, yout in zip(ys, ypred)]

    # 4. 求平均：sum(losses) / 样本数量。
    total = sum(losses) * (1.0 / len(losses))
    return total, ypred


def loss_with_breakdown():
    ypred = [predict(x) for x in xs]
    losses = []

    print('i | x | ygt | yout | diff = yout-ygt | squared')
    print('--|---|-----|------|------------------|--------')

    for i, (xrow, ygt, yout) in enumerate(zip(xs, ys, ypred)):
        diff = yout - ygt
        squared = diff ** 2
        losses.append(squared)
        print(
            f'{i} | {xrow} | {ygt:+.1f} | {yout.data:+.4f} | '
            f'{diff.data:+.4f} | {squared.data:.4f}'
        )

    total = sum(losses) * (1.0 / len(losses))
    print('sum squared errors =', round(sum(item.data for item in losses), 4))
    print('average loss       =', total.data)
    return total, ypred, losses


initial_loss, initial_preds, initial_sample_losses = loss_with_breakdown()
print('initial predictions:', [round(p.data, 4) for p in initial_preds])



### 对照代码里的 4 行核心逻辑

现在再看紧凑版，就不神秘了：

```python
ypred = [predict(x) for x in xs]
```

意思是：4 条输入各预测一次。

```python
losses = [(yout - ygt) ** 2 for ygt, yout in zip(ys, ypred)]
```

意思是：标准答案和预测值一一配对，每条都算平方误差。

```python
total = sum(losses) * (1.0 / len(losses))
```

意思是：把每条的平方误差加起来，再除以样本数量，得到平均 loss。

```python
return total, ypred
```

意思是：返回总 loss，同时也把预测列表返回，方便我们观察模型现在猜成什么样。



## 4. 单步训练拆开看

一轮训练只有四步：

```text
1. forward：算 loss
2. zero_grad：清空旧梯度
3. backward：算新梯度
4. update：p.data += -learning_rate * p.grad
```

为什么要 `zero_grad`？因为 micrograd 里的梯度会用 `+=` 累加，不清空就会混入上一轮。

In [ ]:
learning_rate = 0.05

current_loss, _ = loss()
model.zero_grad()
current_loss.backward()

print('before update loss:', current_loss.data)
print('first param before:', model.parameters()[0])

for p in model.parameters():
    p.data += -learning_rate * p.grad

new_loss, _ = loss()
print('first param after :', model.parameters()[0])
print('after update loss :', new_loss.data)

## 5. 完整训练循环

下面跑 40 步。loss 不一定每一步都严格下降，但整体应该往下。

In [ ]:
random.seed(1337)
model = MLP(2, [4, 1])
learning_rate = 0.1
history = []

for step in range(40):
    total_loss, ypred = loss()
    history.append(total_loss.data)

    model.zero_grad()
    total_loss.backward()

    for p in model.parameters():
        p.data += -learning_rate * p.grad

    if step % 5 == 0 or step == 39:
        print(f'step {step:02d} loss={total_loss.data:.6f} predictions={[round(p.data, 3) for p in ypred]}')

final_loss, final_preds = loss()
print()
print('all history', history)
print()
print('initial loss:', history[0])
print('final loss  :', final_loss.data)
assert final_loss.data < history[0]
print('OK: final loss is lower than initial loss.')

## 6. 改一行实验

你现在可以只改一个东西：

```text
learning_rate = 0.01
learning_rate = 0.1
learning_rate = 1.0
```

观察：

```text
太小：下降很慢
合适：比较稳定下降
太大：可能震荡甚至变差
```

这就是学习率的直觉。

## What To Remember

训练循环只记住六句话：

```text
1. model(x) 做前向预测。
2. loss 衡量预测和目标差多少。
3. zero_grad 清空上一轮梯度。
4. loss.backward() 计算每个参数的 grad。
5. 参数更新用 p.data += -learning_rate * p.grad。
6. 反复做很多轮，目标是让 loss 下降。
```

你前面学的所有导数、链式法则、Value、backward，最后都服务于第 4 步。

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - 一步更新

In [ ]:
# p=5, grad=8, lr=0.1
student_new_p = None



qa_check('qa_check_class_06_predict', globals())

<details><summary>Show / Hide 本题提示</summary>

- 按训练循环顺序想：先清零 grad，再算 loss，再 backward，最后 `p.data -= learning_rate * p.grad`。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_new_p` 约等于 `4.2`。

</details>

## Run - 训练一个 y=2x+1 的一参数模型

In [ ]:
from micrograd.engine import Value

xs = [0.0, 1.0, 2.0, 3.0]
ys = [1.0, 3.0, 5.0, 7.0]
w = Value(0.0)
b = Value(0.0)

def predict(x):
    return w*x + b

def loss():
    losses = [(predict(x)-y)**2 for x, y in zip(xs, ys)]
    return sum(losses) * (1.0 / len(losses))

before = loss().data
for p in [w, b]:
    p.grad = 0
L = loss()
L.backward()
for p in [w, b]:
    p.data -= 0.1 * p.grad
after = loss().data
print('loss before/after:', before, after)
print('w,b:', w, b)

## Modify - 改 learning_rate

In [ ]:
# old=5, grad=8, lr=0.01
student_new_p_small_lr = None



qa_check('qa_check_class_06_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 按训练循环顺序想：先清零 grad，再算 loss，再 backward，最后 `p.data -= learning_rate * p.grad`。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_new_p_small_lr` 约等于 `4.92`。

</details>